# Training Strategy
1. Train/validation/test split
2. Time-based split
3. Data leakage prevention
4. Class imbalance check
5. Sampling/class weights/threshold decision

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import csr_matrix
from scipy.sparse import hstack

In [2]:
file_path = "../data/processed/avazu_eda_ready.parquet"
data = pd.read_parquet(file_path)
print("Dataset loaded successfully")

Dataset loaded successfully


In [3]:
data.head()

,click,C1,banner_pos,site_id,site_domain,site_category,app_id,app_domain,app_category,device_id,...,C18,C19,C20,C21,timestamp,day,hour_of_day,day_of_week,is_weekend,time_period
0,0,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,...,0,35,-1,79,2014-10-21,21,0,1,0,night
1,0,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,...,0,35,100084,79,2014-10-21,21,0,1,0,night
2,0,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,...,0,35,100084,79,2014-10-21,21,0,1,0,night
3,0,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,07d7df22,a99f214a,...,0,35,100084,79,2014-10-21,21,0,1,0,night
4,0,1005,1,fe8cc448,9166c161,0569f928,ecad2386,7801e8d9,07d7df22,a99f214a,...,0,35,-1,157,2014-10-21,21,0,1,0,night


In [4]:
data.shape

(40428967, 28)

In [5]:
# Check the Timestamp
data["timestamp"].head()

0   2014-10-21
1   2014-10-21
2   2014-10-21
3   2014-10-21
4   2014-10-21
Name: timestamp, dtype: datetime64[us]

In [6]:
# Sort the Dataset by Time
data = data.sort_values("timestamp")

In [7]:
data["timestamp"].head()

0       2014-10-21
79344   2014-10-21
79343   2014-10-21
79342   2014-10-21
79341   2014-10-21
Name: timestamp, dtype: datetime64[us]

In [8]:
data["timestamp"].tail()

40373224   2014-10-30 23:00:00
40373223   2014-10-30 23:00:00
40373222   2014-10-30 23:00:00
40373240   2014-10-30 23:00:00
40428966   2014-10-30 23:00:00
Name: timestamp, dtype: datetime64[us]

In [9]:
# check the values move in increasing order.
data["timestamp"].is_monotonic_increasing

True

In [10]:
# Define the Train, Validation, and Test Split
total_rows = len(data)
train_end = int(total_rows * 0.70)
validation_end = int(total_rows * 0.85)

In [11]:
print("Total rows:", total_rows)
print("Training end position:", train_end)
print("Validation end position:", validation_end)

Total rows: 40428967
Training end position: 28300276
Validation end position: 34364621


# Create Training, Validation and Test Data

In [12]:
# iloc selects rows based on their position.
train_data = data.iloc[:train_end].copy()
validation_data = data.iloc[train_end:validation_end].copy()
test_data = data.iloc[validation_end:].copy()

In [13]:
# Check Split Sizes
print("Training rows:", len(train_data))
print("Validation rows:", len(validation_data))
print("Test rows:", len(test_data))

Training rows: 28300276
Validation rows: 6064345
Test rows: 6064346


In [14]:
# Check percentages of splitted data
train_percentage = (len(train_data) / len(data) * 100)
validation_percentage = (len(validation_data) / len(data) * 100)
test_percentage = (len(test_data) / len(data) * 100)

In [15]:
print(f"Training: {train_percentage:.2f}%")
print(f"Validation: {validation_percentage:.2f}%")
print(f"Test: {test_percentage:.2f}%")

Training: 70.00%
Validation: 15.00%
Test: 15.00%


In [16]:
# Check Time Ranges
# TRAINING DATA
print("TRAINING DATA")
print("Start:", train_data["timestamp"].min())
print("End:", train_data["timestamp"].max())

TRAINING DATA
Start: 2014-10-21 00:00:00
End: 2014-10-28 08:00:00


In [17]:
# VALIDATION DATA
print("VALIDATION DATA")
print("Start:", validation_data["timestamp"].min())
print("End:", validation_data["timestamp"].max())

VALIDATION DATA
Start: 2014-10-28 08:00:00
End: 2014-10-29 11:00:00


In [18]:
# TEST DATA
print("TEST DATA")
print("Start:", test_data["timestamp"].min())
print("End:", test_data["timestamp"].max())

TEST DATA
Start: 2014-10-29 11:00:00
End: 2014-10-30 23:00:00


# Check Class Imbalance

In [19]:
train_data["click"].value_counts()

click
0    23346931
1     4953345
Name: count, dtype: int64

In [20]:
train_class_percentage = (train_data["click"].value_counts(normalize=True)* 100)
train_class_percentage

click
0    82.497185
1    17.502815
Name: proportion, dtype: float64

In [21]:
# Calculate CTR for Each Dataset
train_ctr = train_data["click"].mean()
validation_ctr = validation_data["click"].mean()
test_ctr = test_data["click"].mean()

In [22]:
print(f"Training CTR: {train_ctr * 100:.2f}%")
print(f"Validation CTR: "f"{validation_ctr * 100:.2f}%")
print(f"Test CTR: {test_ctr * 100:.2f}%")

Training CTR: 17.50%
Validation CTR: 14.88%
Test CTR: 16.64%


# Separate Features and Target

In [23]:
# Train 
X_train = train_data.drop(columns=["click"])
y_train = train_data["click"]

In [24]:
# Validation
X_validation = validation_data.drop(columns=["click"])
y_validation = validation_data["click"]

In [25]:
# Test
X_test = test_data.drop(columns=["click"])
y_test = test_data["click"]

In [26]:
data.columns.tolist()

['click',
 'C1',
 'banner_pos',
 'site_id',
 'site_domain',
 'site_category',
 'app_id',
 'app_domain',
 'app_category',
 'device_id',
 'device_ip',
 'device_model',
 'device_type',
 'device_conn_type',
 'C14',
 'C15',
 'C16',
 'C17',
 'C18',
 'C19',
 'C20',
 'C21',
 'timestamp',
 'day',
 'hour_of_day',
 'day_of_week',
 'is_weekend',
 'time_period']

In [27]:
X_train = X_train.drop(columns=["timestamp"])
X_validation = X_validation.drop(columns=["timestamp"])
X_test = X_test.drop(columns=["timestamp"])

In [28]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (28300276, 26)
y_train: (28300276,)


In [29]:
# Define the frequency encoding columns
frequency_columns = ["device_ip","device_id","app_id","device_model","site_domain","site_id","C14","app_domain","C17","C20"]

In [30]:
# Create a dictionary to store frequency mappings
# We need to keep these mappings because later the same mappings must be applied to validation and test data.
frequency_mappings = {}

In [31]:
X_train["device_id"].head(10)

0        a99f214a
79344    aa48d0c8
79343    a99f214a
79342    a99f214a
79341    a99f214a
79340    a99f214a
79339    a99f214a
79338    40e9024f
79337    a99f214a
79336    a99f214a
Name: device_id, dtype: str

In [32]:
device_id_counts = X_train["device_id"].value_counts()

In [33]:
device_id_counts.head(10)

device_id
a99f214a    23336464
c357dbff       19667
936e92fb       11107
afeffc18        4839
d857ffbb        3260
cef4c8cc        3002
28dc8687        2904
987552d1        2903
b09da1c4        2395
03559b29        1606
Name: count, dtype: int64

In [34]:
device_id_frequency_map = (X_train["device_id"].value_counts(normalize=True))

In [35]:
device_id_frequency_map.head(10)

device_id
a99f214a    0.824602
c357dbff    0.000695
936e92fb    0.000392
afeffc18    0.000171
d857ffbb    0.000115
cef4c8cc    0.000106
28dc8687    0.000103
987552d1    0.000103
b09da1c4    0.000085
03559b29    0.000057
Name: proportion, dtype: float64

In [36]:
frequency_mappings["device_id"] = device_id_frequency_map

In [37]:
frequency_mappings.keys()

dict_keys(['device_id'])

In [38]:
# Create device_id_freq in training data
X_train["device_id_freq"] = (X_train["device_id"].map(device_id_frequency_map))

In [39]:
X_train[["device_id", "device_id_freq"]].head(10)

,device_id,device_id_freq
0,a99f214a,8.246020e-01
79344,aa48d0c8,3.180181e-07
79343,a99f214a,8.246020e-01
79342,a99f214a,8.246020e-01
79341,a99f214a,8.246020e-01
79340,a99f214a,8.246020e-01
79339,a99f214a,8.246020e-01
79338,40e9024f,3.533534e-08
79337,a99f214a,8.246020e-01
79336,a99f214a,8.246020e-01


In [40]:
# Apply the map to validation data
X_validation["device_id_freq"] = (X_validation["device_id"].map(device_id_frequency_map).fillna(0))

In [41]:
# Apply the training map to test data
X_test["device_id_freq"] = (X_test["device_id"].map(device_id_frequency_map) .fillna(0))

In [42]:
X_train[["device_id", "device_id_freq"]].head()

,device_id,device_id_freq
0,a99f214a,8.246020e-01
79344,aa48d0c8,3.180181e-07
79343,a99f214a,8.246020e-01
79342,a99f214a,8.246020e-01
79341,a99f214a,8.246020e-01


In [43]:
X_validation[["device_id", "device_id_freq"]].head()

,device_id,device_id_freq
28494745,a99f214a,0.824602
28494744,a99f214a,0.824602
28494743,a99f214a,0.824602
28494724,a99f214a,0.824602
28494723,a99f214a,0.824602


In [44]:
X_test[["device_id", "device_id_freq"]].head()

,device_id,device_id_freq
34310771,a99f214a,0.824602
34310772,d259ba42,0.000000
34310773,a99f214a,0.824602
34310774,a99f214a,0.824602
34310775,a99f214a,0.824602


In [45]:
# This creates a training frequency map for each column.
for column in frequency_columns:
     frequency_map = (X_train[column].value_counts(normalize=True))
print(column)
print(frequency_map.head())

C20
C20
-1         0.466963
 100084    0.063521
 100111    0.048732
 100148    0.041847
 100077    0.041305
Name: proportion, dtype: float64


In [46]:
frequency_mappings = {}
for column in frequency_columns:
    frequency_map = (X_train[column].value_counts(normalize=True)
                    )
    frequency_mappings[column] = frequency_map

In [47]:
frequency_mappings.keys()

dict_keys(['device_ip', 'device_id', 'app_id', 'device_model', 'site_domain', 'site_id', 'C14', 'app_domain', 'C17', 'C20'])

In [48]:
# Encode training data only
for column in frequency_columns:
    frequency_map = (X_train[column].value_counts(normalize=True)
                    )

    frequency_mappings[column] = frequency_map
    new_column = column + "_freq"
    X_train[new_column] = (X_train[column].map(frequency_map))

In [49]:
# Add validation
frequency_mappings = {}
for column in frequency_columns:
    frequency_map = (X_train[column].value_counts(normalize=True)
                    )
    frequency_mappings[column] = frequency_map
    new_column = column + "_freq"
    X_train[new_column] = (X_train[column].map(frequency_map)
                          )
    X_validation[new_column] = (X_validation[column].map(frequency_map).fillna(0))

In [50]:
# Add test
frequency_mappings = {}
for column in frequency_columns:
    print("Encoding:", column)
    frequency_map = (X_train[column].value_counts(normalize=True))
    frequency_mappings[column] = frequency_map
    new_column = column + "_freq"
    X_train[new_column] = (X_train[column].map(frequency_map))
    X_validation[new_column] = (X_validation[column].map(frequency_map).fillna(0))
    X_test[new_column] = (X_test[column].map(frequency_map).fillna(0))

    print("Created:", new_column)
    print()

Encoding: device_ip
Created: device_ip_freq

Encoding: device_id
Created: device_id_freq

Encoding: app_id
Created: app_id_freq

Encoding: device_model
Created: device_model_freq

Encoding: site_domain
Created: site_domain_freq

Encoding: site_id
Created: site_id_freq

Encoding: C14
Created: C14_freq

Encoding: app_domain
Created: app_domain_freq

Encoding: C17
Created: C17_freq

Encoding: C20
Created: C20_freq



In [51]:
frequency_encoded_columns = [
    column + "_freq"
    for column in frequency_columns
]
frequency_encoded_columns

['device_ip_freq',
 'device_id_freq',
 'app_id_freq',
 'device_model_freq',
 'site_domain_freq',
 'site_id_freq',
 'C14_freq',
 'app_domain_freq',
 'C17_freq',
 'C20_freq']

In [52]:
X_train[frequency_encoded_columns].head()

,device_ip_freq,device_id_freq,app_id_freq,device_model_freq,site_domain_freq,site_id_freq,C14_freq,app_domain_freq,C17_freq,C20_freq
0,1.819912e-03,8.246020e-01,0.657962,0.000583,0.173300,0.173300,0.014732,0.695755,0.139321,0.466963
79344,1.227197e-04,3.180181e-07,0.001381,0.009346,0.355104,0.342038,0.004268,0.695755,0.004268,0.466963
79343,5.357757e-03,8.246020e-01,0.657962,0.004185,0.173300,0.173300,0.015525,0.695755,0.139321,0.466963
79342,3.533534e-08,8.246020e-01,0.001479,0.000113,0.355104,0.342038,0.001250,0.107414,0.001250,0.001297
79341,3.533534e-08,8.246020e-01,0.657962,0.004286,0.000183,0.000158,0.009289,0.695755,0.009289,0.039648


In [53]:
X_train[["device_id", "device_id_freq"]].head(10)

,device_id,device_id_freq
0,a99f214a,8.246020e-01
79344,aa48d0c8,3.180181e-07
79343,a99f214a,8.246020e-01
79342,a99f214a,8.246020e-01
79341,a99f214a,8.246020e-01
79340,a99f214a,8.246020e-01
79339,a99f214a,8.246020e-01
79338,40e9024f,3.533534e-08
79337,a99f214a,8.246020e-01
79336,a99f214a,8.246020e-01


In [54]:
# Check missing values in encoded columns
X_train[frequency_encoded_columns].isnull().sum()

device_ip_freq       0
device_id_freq       0
app_id_freq          0
device_model_freq    0
site_domain_freq     0
site_id_freq         0
C14_freq             0
app_domain_freq      0
C17_freq             0
C20_freq             0
dtype: int64

In [55]:
X_validation[frequency_encoded_columns].isnull().sum()

device_ip_freq       0
device_id_freq       0
app_id_freq          0
device_model_freq    0
site_domain_freq     0
site_id_freq         0
C14_freq             0
app_domain_freq      0
C17_freq             0
C20_freq             0
dtype: int64

In [56]:
X_test[frequency_encoded_columns].isnull().sum()

device_ip_freq       0
device_id_freq       0
app_id_freq          0
device_model_freq    0
site_domain_freq     0
site_id_freq         0
C14_freq             0
app_domain_freq      0
C17_freq             0
C20_freq             0
dtype: int64

In [57]:
# Remove the original frequency-encoded columns
X_train = X_train.drop(columns=frequency_columns)

X_validation = X_validation.drop(columns=frequency_columns)

X_test = X_test.drop(columns=frequency_columns)

In [58]:
X_train.columns.tolist()

['C1',
 'banner_pos',
 'site_category',
 'app_category',
 'device_type',
 'device_conn_type',
 'C15',
 'C16',
 'C18',
 'C19',
 'C21',
 'day',
 'hour_of_day',
 'day_of_week',
 'is_weekend',
 'time_period',
 'device_id_freq',
 'device_ip_freq',
 'app_id_freq',
 'device_model_freq',
 'site_domain_freq',
 'site_id_freq',
 'C14_freq',
 'app_domain_freq',
 'C17_freq',
 'C20_freq']

In [59]:
X_train.dtypes

C1                      int64
banner_pos              int64
site_category             str
app_category              str
device_type             int64
device_conn_type        int64
C15                     int64
C16                     int64
C18                     int64
C19                     int64
C21                     int64
day                      int8
hour_of_day              int8
day_of_week              int8
is_weekend               int8
time_period          category
device_id_freq        float64
device_ip_freq        float64
app_id_freq           float64
device_model_freq     float64
site_domain_freq      float64
site_id_freq          float64
C14_freq              float64
app_domain_freq       float64
C17_freq              float64
C20_freq              float64
dtype: object

In [61]:
remaining_columns = (X_train.select_dtypes(include=["object","string","category"]).columns.tolist())
remaining_columns

['site_category', 'app_category', 'time_period']

In [62]:
# Frequency encode the remaining 3 columns
for column in remaining_columns:

    print("Encoding:", column)

    frequency_map = (
        X_train[column]
        .value_counts(normalize=True)
    )

    frequency_mappings[column] = frequency_map

    new_column = column + "_freq"

    X_train[new_column] = (
        X_train[column]
        .map(frequency_map)
        .fillna(0)
    )

    X_validation[new_column] = (
        X_validation[column]
        .map(frequency_map)
        .fillna(0)
    )

    X_test[new_column] = (
        X_test[column]
        .map(frequency_map)
        .fillna(0)
    )

    print("Created:", new_column)
    print()

Encoding: site_category
Created: site_category_freq

Encoding: app_category
Created: app_category_freq

Encoding: time_period
Created: time_period_freq



In [63]:
# Check the encoded values
X_train[
    [
        "site_category",
        "site_category_freq",
        "app_category",
        "app_category_freq",
        "time_period",
        "time_period_freq"
    ]
].head()

,site_category,site_category_freq,app_category,app_category_freq,time_period,time_period_freq
0,28905ebd,0.196165,07d7df22,0.667639,night,0.228839
79344,50e219e0,0.391277,0f2161f8,0.232274,night,0.228839
79343,28905ebd,0.196165,07d7df22,0.667639,night,0.228839
79342,50e219e0,0.391277,0f2161f8,0.232274,night,0.228839
79341,3e814130,0.082635,07d7df22,0.667639,night,0.228839


In [64]:
# Remove the original text columns
X_train = X_train.drop(
    columns=remaining_columns
)

X_validation = X_validation.drop(
    columns=remaining_columns
)

X_test = X_test.drop(
    columns=remaining_columns
)

In [65]:
# Check whether text columns remain
X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

['time_period_freq']

In [66]:
X_train["time_period_freq"] = X_train["time_period_freq"].astype("float32")

X_validation["time_period_freq"] = X_validation["time_period_freq"].astype("float32")

X_test["time_period_freq"] = X_test["time_period_freq"].astype("float32")

In [67]:
X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

[]

In [68]:
#Check your final columns
X_train.columns.tolist()

['C1',
 'banner_pos',
 'device_type',
 'device_conn_type',
 'C15',
 'C16',
 'C18',
 'C19',
 'C21',
 'day',
 'hour_of_day',
 'day_of_week',
 'is_weekend',
 'device_id_freq',
 'device_ip_freq',
 'app_id_freq',
 'device_model_freq',
 'site_domain_freq',
 'site_id_freq',
 'C14_freq',
 'app_domain_freq',
 'C17_freq',
 'C20_freq',
 'site_category_freq',
 'app_category_freq',
 'time_period_freq']

In [69]:
X_train.shape

(28300276, 26)

In [70]:
import os
os.makedirs("../data/processed", exist_ok=True)

In [71]:
# Save Test data
X_train.to_parquet(
    "../data/processed/X_train.parquet",
    index=False
)

y_train.to_frame().to_parquet(
    "../data/processed/y_train.parquet",
    index=False
)

In [72]:
# Save validation data
X_validation.to_parquet(
    "../data/processed/X_validation.parquet",
    index=False
)

y_validation.to_frame().to_parquet(
    "../data/processed/y_validation.parquet",
    index=False
)

In [73]:
# Save test data
X_test.to_parquet(
    "../data/processed/X_test.parquet",
    index=False
)

y_test.to_frame().to_parquet(
    "../data/processed/y_test.parquet",
    index=False
)